# Day 14: Pandas 进阶 —— 习题

> **范围**: 字符串方法、apply、缺失值、分箱、透视表
> **数据**: `../data/sales.csv`（500行，9列）
> **建议用时**: 60-90 分钟
> **提示**: 能用 `.str` / `np.where` / `fillna` 解决就别写循环

## Easy

**1. 字符串方法 —— 筛选与替换**

基于 `df = pd.read_csv("../data/sales.csv")`：
- 筛选出 `product` 列包含 `"Phone"` 的所有订单，统计数量
- 筛选出 `order_id` 以 `"O10"` 开头的订单，查看前5行
- 把 `country` 列中的 `"US"` 全部替换为 `"USA"`（原地修改）
- 创建新列 `product_short` = `product` 的前3个字符（提示：`.str[:3]`）

In [13]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/sales.csv")

print(df['product'].str.contains('Phone', case=False).sum())

print(df[df['order_id'].str.startswith('O10')].head(5))

print(df['country'].str.replace('US', 'USA').unique())

df['product_short'] = df['product'].str[:3]
print(df['product_short'].unique())

160
  order_id customer_id     product   category  quantity  price  order_date  \
0    O1000        C007    Keyboard  Accessory         2   1299  2024-01-01   
1    O1001        C004    Keyboard  Accessory         1     99  2024-01-01   
2    O1002        C005      Laptop   Computer         4     99  2024-01-02   
3    O1003        C007  Headphones      Audio         4     99  2024-01-03   
4    O1004        C003       Phone     Mobile         5     99  2024-01-03   

   country  total  
0  Germany   2598  
1       US     99  
2       US    396  
3       US    396  
4   France    495  
['Germany' 'USA' 'France' 'UK' 'China']
['Key' 'Lap' 'Hea' 'Pho' 'Mon' 'Mou']


参考答案

问题: .str.replace 返回的是新 Series，你没有赋值回 df['country']，所以原 DataFrame 的 country 列没有被修改。你只是打印了替换后的结果。

正确做法:

In [65]:
df['country'] = df['country'].str.replace('US', 'USA')
print(df['country'].unique())

['Germany' 'USA' 'France' 'UK' 'China']


这是 Pandas 的「不可变性」陷阱: str.replace() / fillna() / sort_values() 等大多数方法默认返回新对象，不修改原 DataFrame。要原地修改必须显式赋值。

**2. 缺失值处理 —— fillna 与 dropna**

基于 `df`：
- 检查每列的缺失值数量（`.isnull().sum()`）
- 假设 `total` 列有缺失值，用该列的**均值**填充，存为新列 `total_filled`
  （提示：先人为制造几个 NaN 来练习：`df.loc[df.sample(5).index, "total"] = np.nan`）
- 验证填充后 `total_filled` 没有缺失值
- 对 `country` 列，如果某行缺失则填充为 `"Unknown"`

In [18]:
print(df.isnull().sum())

df.loc[df.sample(5).index, "total"] = np.nan
df['total_filled'] = df['total'].fillna(df['total'].mean())

print(df['total_filled'].isnull().sum())

print(df['country'].fillna('Unknown'))

order_id          0
customer_id       0
product           0
category          0
quantity          0
price             0
order_date        0
country           0
total            10
product_short     0
total_filled      0
dtype: int64
0
0      Germany
1           US
2           US
3           US
4       France
        ...   
495         UK
496         UK
497     France
498     France
499         UK
Name: country, Length: 500, dtype: object


参考答案

问题: 同上。fillna 返回新 Series，没有赋值回 df['country']。df['country'] 本身没有变化。

正确做法:

In [66]:
df['country'] = df['country'].fillna('Unknown')

**3. 条件赋值 —— np.where 与 where**

基于 `df`：
- 创建 `total_level`：如果 `total` >= 5000 为 `"高"`，>= 2000 为 `"中"`，否则为 `"低"`
  （提示：嵌套 `np.where`）
- 用 `.value_counts()` 统计各等级数量
- 用 `df['total'].where(df['total'] <= 3000, 3000)` 创建 `total_cap`：超过3000的截断为3000
- 验证 `total_cap` 的最大值是否等于 3000

In [21]:
df['total_level'] = np.where(df['total']>=5000, '高', np.where(df['total'] >= 2000, '中', '低'))
print(df['total_level'].value_counts())

df['total_cap'] = df['total'].where(df['total'] <= 3000, 3000)
print(df['total_cap'].max())

total_level
低    308
中    109
高     83
Name: count, dtype: int64
3000.0


## Medium

**4. apply —— 自定义函数**

基于 `df`：
- 写函数 `discount_label(row)`：根据 `quantity` 给折扣标签
  - quantity >= 5 → `"大量折扣"`
  - quantity == 4 → `"中等折扣"`
  - quantity <= 3 → `"标准折扣"`
- 用 `df.apply(discount_label, axis=1)` 创建新列 `discount_label`
- 统计每个标签的数量
- **思考**：这个函数能用 `np.where` 重写吗？试着重写

In [24]:
def discount_label(row):
    if row['quantity'] >= 5:
        return '大量折扣'
    elif row['quantity'] == 4:
        return '中等折扣'
    return '标准折扣'

df['discount_label'] = df.apply(discount_label, axis=1)
print(df['discount_label'].value_counts())

df['discount_label2'] = np.where(df['quantity']>=5, '大量折扣', np.where(df['quantity']==4, '中等折扣', '标准折扣'))
print(df['discount_label2'].value_counts())

discount_label
标准折扣    311
大量折扣    100
中等折扣     89
Name: count, dtype: int64
discount_label2
标准折扣    311
大量折扣    100
中等折扣     89
Name: count, dtype: int64


**5. 分箱 —— cut 与 qcut**

基于 `df`：
- 用 `pd.cut` 把 `total` 分成4个等宽区间：`[0,1000,3000,6000,10000]`，标签为 `["低","中","高","超高"]`
- 统计每个区间的订单数量
- 用 `pd.qcut` 把 `total` 分成4个等分位箱，标签为 `["Q1","Q2","Q3","Q4"]`
- 统计每个分位箱的订单数量（应该大致相等）
- 计算每个等宽区间的 `quantity` 平均值（用循环或筛选）

In [27]:
df['total_bin'] = pd.cut(df['total'], bins=[0, 1000, 3000, 6000, 10000], labels=['低','中','高','超高'])
print(df['total_bin'].value_counts())

df['total_q'] = pd.qcut(df['total'], q=4, labels=['Q1','Q2','Q3','Q4'])
print(df['total_q'].value_counts())

print(df.groupby('total_bin')['quantity'].agg('mean'))

total_bin
中     181
低     169
高      87
超高     48
Name: count, dtype: int64
total_q
Q1    137
Q2    125
Q3    114
Q4    109
Name: count, dtype: int64
total_bin
低     2.372781
中     2.839779
高     3.482759
超高    4.750000
Name: quantity, dtype: float64


C:\Users\69261\AppData\Local\Temp\ipykernel_24264\492472653.py:7: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('total_bin')['quantity'].agg('mean'))


**6. 复杂筛选与 loc 赋值**

基于 `df`：
- 筛选出 `country` 在 `["UK","US"]` 中 **且** `product` 包含 `"Laptop"` 或 `"Monitor"` 的订单
  （提示：`product` 用 `.str.contains` + `|` 正则）
- 用 `loc` 把上述筛选结果的 `category` 统一改为 `"Electronics"`
- 验证修改后的 `category` 唯一值
- 把 `df` 中 `total` 最高的 **前 10 名** 的 `country` 改为 `"VIP"`（提示：先排序取索引，再用 loc）

In [ ]:
mask = (df['country'].isin(['UK','US'])) & df['product'].str.contains('Laptop|Monitor')
df.loc[mask,'category'] = 'Electronics'
print(df['category'].value_counts())

df.sort_values('total', ascending=False)
df.loc[0:9,'country'] = 'VIP'
print(df.head(10))

category
Accessory      180
Electronics    100
Mobile          93
Audio           67
Computer        60
Name: count, dtype: int64
  order_id customer_id     product     category  quantity  price  order_date  \
0    O1000        C007    Keyboard    Accessory         2   1299  2024-01-01   
1    O1001        C004    Keyboard    Accessory         1     99  2024-01-01   
2    O1002        C005      Laptop  Electronics         4     99  2024-01-02   
3    O1003        C007  Headphones        Audio         4     99  2024-01-03   
4    O1004        C003       Phone       Mobile         5     99  2024-01-03   
5    O1005        C008      Laptop  Electronics         2     99  2024-01-04   
6    O1006        C005       Phone       Mobile         5   1299  2024-01-05   
7    O1007        C005     Monitor  Electronics         2    599  2024-01-06   
8    O1008        C007       Phone       Mobile         5    299  2024-01-06   
9    O1009        C002  Headphones        Audio         3     99  2024

参考答案

问题: df.sort_values() 返回新 DataFrame，默认 inplace=False，原 df 没有排序。所以 df.loc[0:9,'country'] 改的是原 df 的索引 0~9 行，而不是 total 最高的前 10 名。

正确做法:

In [67]:
# 方法1: 先取索引再赋值
top10_indices = df.sort_values('total', ascending=False).head(10).index
df.loc[top10_indices, 'country'] = 'VIP'

# 方法2: 用 nlargest（更简洁）
top10 = df.nlargest(10, 'total')
df.loc[top10.index, 'country'] = 'VIP'

关键教训: Pandas 的 sort_values 默认不修改原 df。loc[0:9] 是按索引标签取的，不是按排序后的位置。如果索引已经被打乱（如筛选后），0:9 可能不是前10行。

**7. 透视表 —— pivot_table**

基于 `df`：
- 创建透视表：行=country，列=category，值=total，聚合=sum，缺失填0
- 找出哪个国家的哪个品类销售额最高（提示：先用透视表，再用 `.stack()` + `.idxmax()`）
- 创建第二个透视表：行=country，值=total，聚合=[sum, mean, count]
- 导出透视表为 CSV 文件 `pivot_summary.csv`

In [45]:
pivot = pd.pivot_table(
    df,
    values='total',
    index='country',
    columns='category',
    aggfunc='sum',
    fill_value=0
)
print(pivot)

print(pivot.stack().idxmax())

pivot2 = pd.pivot_table(
    df,
    values='total',
    index='country',
    aggfunc=['sum', 'mean', 'count'],
    fill_value=0
)
print(pivot2)
pivot2.to_csv('pivot_summary.csv')


category  Accessory    Audio  Computer  Electronics    Mobile
country                                                      
China       21271.0   8191.0   32159.0          0.0   18480.0
France      54139.0  58448.0   51431.0          0.0   41059.0
Germany     42444.0  18678.0   49840.0          0.0   25970.0
UK         161598.0  73734.0       0.0     156705.0  120803.0
US         152627.0  41650.0       0.0      63018.0   31042.0
VIP          2697.0    693.0       0.0       1792.0    8485.0
('UK', 'Accessory')
              sum         mean count
            total        total total
country                             
China     80101.0  2503.156250    32
France   205077.0  2698.381579    76
Germany  136932.0  2208.580645    62
UK       512840.0  2727.872340   188
US       288337.0  2464.418803   117
VIP       13667.0  1366.700000    10


## Hard

**8. 综合清洗管道 —— 字符串 + 缺失值 + 分箱**

基于 `df`：
1. 读取 `../data/sales.csv`
2. 清洗 `product`：
   - 把包含 `"Keyboard"` 或 `"Mouse"` 的 `product` 统一改为 `"Peripheral"`
   - 用 `.str.contains` + `loc` 条件赋值实现
3. 清洗 `total`：
   - 人为制造 10 个 NaN（`df.loc[df.sample(10).index, "total"] = np.nan`）
   - 按 `country` 分组填充缺失的 `total`：每个国家用该国 `total` 的均值填充
   - 提示：先 `groupby('country')['total'].transform('mean')` 得到每行对应的组均值，再用 `fillna`
4. 分箱：`total` 按 `pd.qcut` 分4箱，标签 `["Budget","Standard","Premium","Luxury"]`
5. 统计每个价格等级的订单数量和平均 `quantity`
6. 输出：把清洗后的 DataFrame 写入 `cleaned_sales.csv`（index=False）

In [50]:
df = pd.read_csv('../data/sales.csv')

mask = df['product'].str.contains('Keyboard|Mouse')
df.loc[mask,'product'] = 'Peripheral'

df.loc[df.sample(10).index, "total"] = np.nan
group_mean = df.groupby('country')['total'].transform('mean')
df['total'] = df['total'].fillna(group_mean)

df['total_qcut'] = pd.qcut(df['total'], q=4, labels=["Budget","Standard","Premium","Luxury"])
print(df.groupby('total_qcut')['quantity'].agg(['count', 'mean']))

df.to_csv('cleaned_sales.csv', index=False)

            count      mean
total_qcut                 
Budget        139  2.438849
Standard      124  2.814516
Premium       112  2.651786
Luxury        125  3.992000


C:\Users\69261\AppData\Local\Temp\ipykernel_24264\3622723069.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('total_qcut')['quantity'].agg(['count', 'mean']))


**9. 日期分析 —— 时间窗口**

基于 `df`：
- 把 `order_date` 转为 datetime，创建 `year_month` 列（`YYYY-MM`）
- 筛选出 2024 年 Q1（1-3月）的订单
- 计算 Q1 每个月的订单数量和总销售额
- 找出 Q1 销售额最高的月份
- 用 `np.where` 创建新列 `is_weekend`：判断 `order_date` 是否为周六或周日
  （提示：`.dt.dayofweek >= 5`，周一=0，周日=6）

In [63]:
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
df['year_month'] = df['order_date'].dt.strftime('%Y-%m')

mask = (df['order_date'].dt.year == 2024) & (df['order_date'].dt.month.isin([1,2,3]))
df_q1 = df.loc[mask,:]
print(df_q1.head())

month_stats = df_q1.groupby('year_month')['total'].agg(['count','sum'])
print(month_stats)

print(month_stats['sum'].idxmax())

df['is_weekend'] = np.where(df['order_date'].dt.dayofweek >= 5, 'Yes', 'No')

  order_id customer_id     product   category  quantity  price order_date  \
0    O1000        C007  Peripheral  Accessory         2   1299 2024-01-01   
1    O1001        C004  Peripheral  Accessory         1     99 2024-01-01   
2    O1002        C005      Laptop   Computer         4     99 2024-01-02   
3    O1003        C007  Headphones      Audio         4     99 2024-01-03   
4    O1004        C003       Phone     Mobile         5     99 2024-01-03   

   country   total total_qcut year_month  
0  Germany  2598.0    Premium    2024-01  
1       US    99.0     Budget    2024-01  
2       US   396.0     Budget    2024-01  
3       US   396.0     Budget    2024-01  
4   France   495.0     Budget    2024-01  
            count            sum
year_month                      
2024-01        43   99365.000000
2024-02        40   81795.000000
2024-03        42  135839.062827
2024-03


**10. 综合挑战 —— 数据报告生成**

写一段完整脚本，从 `../data/sales.csv` 生成一份数据质量报告：

**报告内容**（存储在 dict 中，最后转成 JSON 文件 `data_quality_report.json`）：
```python
{
    "file": "../data/sales.csv",
    "total_rows": int,           # 总行数
    "total_cols": int,           # 总列数
    "missing_summary": {         # 每列缺失值数量
        "col1": int,
        "col2": int,
        ...
    },
    "duplicates": int,           # 完全重复的行数（用 drop_duplicates 比较）
    "numeric_stats": {           # 数值列的统计
        "total": {"min": float, "max": float, "mean": float, "median": float},
        "quantity": {"min": int, "max": int, "mean": float, "median": float}
    },
    "category_distribution": {   # 各品类订单数量
        "Audio": int,
        "Computer": int,
        ...
    },
    "country_distribution": {    # 各国家订单数量
        "UK": int,
        "US": int,
        ...
    },
    "price_level_distribution": { # total 的分箱统计（pd.cut 分4箱）
        "低": int,
        "中": int,
        "高": int,
        "超高": int
    }
}
```

**要求**：
- 全程用 Pandas 完成，不用 for 循环（分箱统计除外）
- 缺失值统计用 `.isnull().sum().to_dict()`
- 品类和国家分布用 `.value_counts().to_dict()`
- 输出前打印 JSON 格式确认内容正确

In [64]:
import json

file_path = "../data/sales.csv"
df = pd.read_csv(file_path)

total_rows = df.shape[0]
total_cols = df.shape[1]

missing_summary = df.isnull().sum().to_dict()

unique_df = df.drop_duplicates()
duplicates = total_rows - unique_df.shape[0]

numeric_stats = {}

total_series = df["total"]
numeric_stats["total"] = {
    "min": float(total_series.min()),
    "max": float(total_series.max()),
    "mean": float(total_series.mean()),
    "median": float(total_series.median())
}

qty_series = df["quantity"]
numeric_stats["quantity"] = {
    "min": int(qty_series.min()),
    "max": int(qty_series.max()),
    "mean": float(qty_series.mean()),
    "median": float(qty_series.median())
}


category_distribution = df["category"].value_counts().to_dict()

country_distribution = df["country"].value_counts().to_dict()


bins_label = ["低", "中", "高", "超高"]
df["price_level"] = pd.cut(df["total"], bins=4, labels=bins_label)
price_level_distribution = df["price_level"].value_counts().to_dict()


report = {
    "file": file_path,
    "total_rows": total_rows,
    "total_cols": total_cols,
    "missing_summary": missing_summary,
    "duplicates": duplicates,
    "numeric_stats": numeric_stats,
    "category_distribution": category_distribution,
    "country_distribution": country_distribution,
    "price_level_distribution": price_level_distribution
}

print(json.dumps(report, ensure_ascii=False, indent=4))


with open("data_quality_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=4)

{
    "file": "../data/sales.csv",
    "total_rows": 500,
    "total_cols": 9,
    "missing_summary": {
        "order_id": 0,
        "customer_id": 0,
        "product": 0,
        "category": 0,
        "quantity": 0,
        "price": 0,
        "order_date": 0,
        "country": 0,
        "total": 0
    },
    "duplicates": 0,
    "numeric_stats": {
        "total": {
            "min": 99.0,
            "max": 9995.0,
            "mean": 2535.432,
            "median": 1797.0
        },
        "quantity": {
            "min": 1,
            "max": 5,
            "mean": 2.968,
            "median": 3.0
        }
    },
    "category_distribution": {
        "Accessory": 180,
        "Computer": 160,
        "Mobile": 93,
        "Audio": 67
    },
    "country_distribution": {
        "UK": 197,
        "US": 125,
        "France": 80,
        "Germany": 66,
        "China": 32
    },
    "price_level_distribution": {
        "低": 311,
        "中": 105,
        "高": 56,
       